In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
#from pyspark.sql.functions import row_number,dense_rank,rank

sampleData = (("Olivia", 28, "Sales", 3000),
              ("Harry", 33, "Sales", 4600),
              ("Smith", 40, "Sales", 4100),
              ("Marry", 25, "Finance", 3000),
              ("Henry", 28, "Sales", 3000),
              ("Lars", 46, "Management", 3300),
              ("Jeny", 26, "Finance", 3900),
              ("Aya", 30, "Marketing", 3000),
              ("Omar", 29, "Marketing", 2000),
              ("Johnny", 39, "Sales", 4100)
              )

columns = ["Employee_Name", "Age", "Department", "Salary"]

df = spark.createDataFrame(data=sampleData, schema=columns)

windowPartitionAgg  = Window.partitionBy("Department")
df.withColumn("Avg", F.avg(F.col("Salary")).over(windowPartitionAgg))\
  .withColumn("Sum", F.sum(F.col("Salary")).over(windowPartitionAgg))\
  .withColumn("Min", F.min(F.col("Salary")).over(windowPartitionAgg))\
  .withColumn("Max", F.max(F.col("Salary")).over(windowPartitionAgg)).show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
#from pyspark.sql.functions import row_number,dense_rank,rank

sampleData = (("Olivia", 28, "Sales", 3000),
              ("Harry", 33, "Sales", 4600),
              ("Smith", 40, "Sales", 4100),
              ("Marry", 25, "Finance", 3000),
              ("Henry", 28, "Sales", 3000),
              ("Lars", 46, "Management", 3300),
              ("Jeny", 26, "Finance", 3900),
              ("Aya", 30, "Marketing", 3000),
              ("Omar", 29, "Marketing", 2000),
              ("Johnny", 39, "Sales", 4100)
              )

columns = ["Employee_Name", "Age", "Department", "Salary"]

df = spark.createDataFrame(data=sampleData, schema=columns)

windowPartition = Window.partitionBy("Department").orderBy(F.col("salary").desc())
df = df.withColumn("prev_salary", F.lag("Salary").over(windowPartition))
df = df.withColumn("Next_salary", F.lead("Salary").over(windowPartition))
df.show()

In [0]:
spark.table("workspace.default.emp_dept_join").display()

In [0]:
spark.createDataFrame(data=sampleData, schema=columns)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
#from pyspark.sql.functions import row_number,dense_rank,rank

sampleData = (("Olivia", 28, "Sales", 3000),
              ("Harry", 33, "Sales", 4600),
              ("Smith", 40, "Sales", 4100),
              ("Marry", 25, "Finance", 3000),
              ("Henry", 28, "Sales", 3000),
              ("Lars", 46, "Management", 3300),
              ("Jeny", 26, "Finance", 3900),
              ("Aya", 30, "Marketing", 3000),
              ("Omar", 29, "Marketing", 2000),
              ("Johnny", 39, "Sales", 4100)
              )

columns = ["Employee_Name", "Age", "Department", "Salary"]

df = spark.createDataFrame(data=sampleData, schema=columns)

windowPartition = Window.partitionBy("Department").orderBy(F.col("salary").desc())
df1 = df.withColumn("sum_dept_salary", F.sum("Salary").over(windowPartition))
df.show()
df1.show()

In [0]:
from pyspark.sql.window import Window
from pyspark.sql import functions as F
#from pyspark.sql.functions import row_number,dense_rank,rank

sampleData = (("Olivia", 28, "Sales", 3000),
              ("Harry", 33, "Sales", 4600),
              ("Smith", 40, "Sales", 4100),
              ("Marry", 25, "Finance", 3000),
              ("Henry", 28, "Sales", 3000),
              ("Lars", 46, "Management", 3300),
              ("Jeny", 26, "Finance", 3900),
              ("Aya", 30, "Marketing", 3000),
              ("Omar", 29, "Marketing", 2000),
              ("Johnny", 39, "Sales", 4100)
              )

columns = ["Employee_Name", "Age", "Department", "Salary"]

df = spark.createDataFrame(data=sampleData, schema=columns)

windowPartition = Window.partitionBy("Department").orderBy(F.col("salary").desc())
# df.printSchema()
df1 = df.withColumn("row_number", F.row_number().over(windowPartition)).withColumn("dense_rank", F.dense_rank().over(windowPartition)).withColumn("rank", F.rank().over(windowPartition))
df1.show()
df1.filter('row_number == 2').show()

Read, Write, Join
* Uploading the orders.csv and sales.csv in to Volumes
* Read Orders.csv and primary key is ProductID
* Read products.csv and primary key is ProductID
* use Write to join these two in delta format

In [0]:
from pyspark.sql.functions import col

In [0]:
ordersdf=spark.read.format("csv").\
    option("header", "true").\
    option("inferSchema", "true").\
    load("/Volumes/mycatalog/default/myvolume/orders.csv")
#display(ordersdf)

for c in ordersdf.columns:
  ordersdf = ordersdf.withColumnRenamed(c,f"o_{c}")



In [0]:
#window functions

In [0]:
#optiomazation

In [0]:
ordersdf.where((col("o_ProductID") > 800) & (col("o_ProductID") < 900)).select("o_CustomerID","o_LineItem","o_LineItemTotal").display()

In [0]:
productsdf=spark.read.format("csv").\
    option("header", "true").\
    option("inferSchema", "true").\
    load("/Volumes/mycatalog/default/myvolume/products.csv")
#display(productsdf)

for c in productsdf.columns:
  productsdf = productsdf.withColumnRenamed(c,f"p_{c}")

In [0]:
df = productsdf.alias("pd").join(ordersdf.alias("od"),col("pd.p_ProductID")==col("od.o_ProductID").alias("orders_productid"),"inner").select("od.*","pd.*")

In [0]:
df.write.partitionBy("p_Category").format("delta").\
    mode("overwrite").\
    save("/Volumes/mycatalog/default/myvolume/porders3")

In [0]:
# Note: bucketBy() is not supported for Delta on serverless.
# Use partitionBy() instead:
df.write.partitionBy("p_Category").format("delta").\
    mode("overwrite").\
    save("/Volumes/mycatalog/default/myvolume/porders4")

In [0]:
df.write.format("delta").\
    mode("append").\
    save("/Volumes/mycatalog/default/myvolume/porders")

In [0]:
df.write.format("parquet").\
    mode("append").\
    save("/Volumes/mycatalog/default/myvolume/porders1")

In [0]:
%sql

restore version '/Volumes/mycatalog/default/myvolume/porders1' as of 0

In [0]:
%sql
restore table delta.`/Volumes/mycatalog/default/myvolume/porders` version as of 1

In [0]:
%sql

describe history '/Volumes/mycatalog/default/myvolume/porders'

In [0]:
%sql 
optimize '/Volumes/mycatalog/default/myvolume/porders2'

In [0]:
%fs ls '/Volumes/mycatalog/default/myvolume/porders2'

In [0]:
%fs ls '/Volumes/mycatalog/default/myvolume/porders3'

In [0]:
spark.conf.set("delta.retentionDurationCheck.enabled" , False)

In [0]:
%sql
optimize '/Volumes/mycatalog/default/myvolume/porders' 

In [0]:
%fs ls '/Volumes/mycatalog/default/myvolume/porders'

In [0]:
%fs ls 'dbfs:/Volumes/mycatalog/default/myvolume/porders/_delta_log/'